# バイオ技術 2-3：糖尿病データで回帰分析に挑戦

2-1では、乳がんデータを使って**分類問題**を学びました。

2-2では、MNIST画像を使って**CNNによる画像分類**を体験しました。

この2-3では、糖尿病データを使って、

> **患者の臨床情報から、1年後の糖尿病進行度を予測できるか？**

という**回帰問題**に挑戦します。

今回は、これまでよりも穴埋め問題を多くしています。

---

## 今日のゴール

1. 分類と回帰の違いを説明できる
2. `X`と`y`を自分で作れる
3. Training dataとTest dataに分割できる
4. `fit()`と`predict()`の役割を説明できる
5. MAE、RMSE、R²で回帰モデルを評価できる
6. Linear RegressionとDecision Tree Regressorを比較できる
7. `max_depth`と過学習の関係を考えられる

---

## このNotebookの進め方

`___` と書かれた部分を自分で書き換えてください。

例えば、

```python
model.___(X_train, y_train)
```

なら、2-1で学んだ内容を思い出して、

```python
model.fit(X_train, y_train)
```

のように完成させます。

**まず自分で考える → わからなければ前のNotebookを見る**

という順番で進めてください。


---
## 0. 分類と回帰を復習する

### Classification（分類）

カテゴリーを予測します。

例：

- 良性 / 悪性
- 陽性 / 陰性
- 0 / 1 / 2 / ... / 9

2-1と2-2は分類問題でした。

### Regression（回帰）

連続した数値を予測します。

例：

- 血糖値
- 薬物濃度
- 疾患進行度

今回の2-3は**回帰問題**です。

### 確認

今回予測するのは、

- カテゴリーですか？
- 数値ですか？

答え：


---
## 1. ライブラリを読み込む

今回はscikit-learnに付属しているDiabetes datasetを使います。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries loaded.")


---
## 2. Diabetes datasetを読み込む

このデータには、

- 442人のデータ
- 10個の説明変数
- 1年後の糖尿病進行度を表す目的変数

が含まれています。

説明変数には、年齢、性別、BMI、血圧、血清測定値などがあります。


In [ ]:
diabetes = load_diabetes(
    as_frame=True
)

df = diabetes.frame.copy()

print("Data shape:", df.shape)
display(df.head())


### データの大きさを確認する

- 行は何人分ですか？
- 説明変数はいくつありますか？
- 目的変数の列名は何ですか？

答え：

- サンプル数：
- 説明変数数：
- 目的変数：


In [ ]:
print("Feature names:")
print(diabetes.feature_names)

print()
print("Columns:")
print(df.columns.tolist())


---
## 3. 目的変数の分布を見る

まず、予測したい`target`がどのような分布をしているか確認します。


In [ ]:
plt.figure(figsize=(7, 5))

sns.histplot(
    data=df,
    x="target",
    bins=30,
    kde=True
)

plt.xlabel("Disease progression")
plt.ylabel("Count")
plt.title("Distribution of Target")
plt.show()


### 考えてみよう

1. targetは0と1だけですか？
2. 連続した数値として分布していますか？
3. なぜ今回はClassificationではなくRegressionなのでしょうか？

答え：

1.
2.
3.


---
## 4. 特徴量とtargetの関係を見る

各特徴量とtargetの相関係数を計算します。


In [ ]:
correlations = (
    df.corr(numeric_only=True)["target"]
    .drop("target")
    .sort_values(
        key=np.abs,
        ascending=False
    )
)

display(
    correlations.to_frame("correlation")
)


In [ ]:
plt.figure(figsize=(8, 5))

correlations.sort_values().plot(
    kind="barh"
)

plt.xlabel("Correlation with target")
plt.title("Feature Correlations")
plt.show()


### ミニ演習1：結果を予想して考える

相関係数の絶対値が最も大きい特徴量は何でしたか？

答え：

その特徴量は、targetと正の相関ですか、負の相関ですか？

答え：

### 注意

相関が強いことは、

> その特徴量が糖尿病進行の原因である

ことを意味するわけではありません。

相関と因果関係は別です。


---
# Part 1：Linear Regression

まずLinear Regressionを使います。

Linear Regressionは、

> 複数の特徴量を組み合わせて、連続した数値を予測する

基本的な回帰モデルです。

ここから穴埋め問題が増えます。


---
## 5. 穴埋め1：Xとyを作る

2-1を思い出してください。

- `X`：説明変数
- `y`：目的変数

です。

`___`を埋めてください。

### ヒント

目的変数の列名は`target`です。


In [ ]:
# 穴埋め問題

X = df.drop(
    columns=[___]
)

y = df[___]

print("X shape:", X.shape)
print("y shape:", y.shape)


正しく実行できたら確認してください。

- Xの列数は10ですか？
- yは1次元ですか？


---
## 6. 穴埋め2：Training dataとTest dataに分ける

2-1と同じように、

- 70%：Training data
- 30%：Test data

に分けます。

`___`を埋めてください。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    ___,
    ___,
    test_size=___,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


### 確認

なぜ、Training dataとTest dataを分ける必要がありますか？

答え：


---
## 7. 穴埋め3：Linear Regressionを学習する

機械学習モデルでは、

```text
model.fit(...)
```

で学習します。

`___`を埋めてください。


In [ ]:
linear_model = LinearRegression()

linear_model.___(
    X_train,
    y_train
)

print("Training complete.")


---
## 8. 穴埋め4：Test dataを予測する

学習したモデルを使って予測します。

`___`を埋めてください。


In [ ]:
linear_pred = linear_model.___(
    X_test
)

print("Number of predictions:", len(linear_pred))
print("First 5 predictions:")
print(linear_pred[:5])


### 考えてみよう

`fit()`と`predict()`の違いを、自分の言葉で説明してください。

- `fit()`：
- `predict()`：


---
## 9. 穴埋め5：回帰モデルを評価する

分類ではAccuracyを使いました。

回帰では、代表的に次の指標を使います。

### MAE

予測値と実測値の誤差の絶対値を平均したものです。

**小さいほど良い**指標です。

### RMSE

大きな誤差をより強く反映する指標です。

**小さいほど良い**指標です。

### R²

モデルが目的変数のばらつきをどの程度説明できたかを表します。

一般的には**大きいほど良い**指標です。

`___`を埋めてください。


In [ ]:
linear_mae = mean_absolute_error(
    ___,
    ___
)

linear_rmse = np.sqrt(
    mean_squared_error(
        ___,
        ___
    )
)

linear_r2 = r2_score(
    ___,
    ___
)

print(f"MAE : {linear_mae:.3f}")
print(f"RMSE: {linear_rmse:.3f}")
print(f"R2  : {linear_r2:.3f}")


### 確認問題

次のうち、値が小さい方が良い指標はどれですか？

- MAE
- RMSE
- R²

答え：


---
## 10. 実測値と予測値を比較する

横軸にActual value、縦軸にPredicted valueを表示します。

良い予測なら、点は対角線付近に集まります。


In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(
    y_test,
    linear_pred,
    alpha=0.7
)

min_value = min(
    y_test.min(),
    linear_pred.min()
)

max_value = max(
    y_test.max(),
    linear_pred.max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Linear Regression: Actual vs Predicted")
plt.show()


### 考えてみよう

1. すべての点が対角線上にありますか？
2. 大きく外れた症例はありますか？
3. Test dataの予測は完全ではないことを確認できましたか？

答え：

1.
2.
3.


---
# Part 2：Decision Tree Regressor

2-1ではDecision Tree Classifierを使って、

> 良性 / 悪性

を分類しました。

Decision TreeはRegressionにも使えます。

今回は、

> 糖尿病進行度という数値

を予測します。


---
## 11. 穴埋め6：Decision Tree Regressorを作る

`max_depth=3`のDecision Tree Regressorを作ります。

`___`を埋めてください。


In [ ]:
tree_model = DecisionTreeRegressor(
    max_depth=___,
    random_state=42
)

tree_model.___(
    X_train,
    y_train
)

tree_pred = tree_model.___(
    X_test
)

print("Decision Tree prediction complete.")


---
## 12. 穴埋め7：Decision Treeを評価する

Linear Regressionと同じ指標で評価します。


In [ ]:
tree_mae = mean_absolute_error(
    ___,
    ___
)

tree_rmse = np.sqrt(
    mean_squared_error(
        ___,
        ___
    )
)

tree_r2 = r2_score(
    ___,
    ___
)

print(f"MAE : {tree_mae:.3f}")
print(f"RMSE: {tree_rmse:.3f}")
print(f"R2  : {tree_r2:.3f}")


---
## 13. 2つのモデルを比較する

Linear RegressionとDecision Treeを比較します。


In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree"
    ],
    "MAE": [
        linear_mae,
        tree_mae
    ],
    "RMSE": [
        linear_rmse,
        tree_rmse
    ],
    "R2": [
        linear_r2,
        tree_r2
    ]
})

display(comparison_df)


### ミニ演習2：モデルを選ぶ

結果を見て答えてください。

1. MAEが小さいのはどちらですか？
2. RMSEが小さいのはどちらですか？
3. R²が大きいのはどちらですか？
4. あなたならどちらのモデルを採用しますか？

答え：

1.
2.
3.
4.


---
## 14. max_depthを自分で変える

2-1では、Decision Treeを深くしすぎると、
Training dataに過剰に適合する可能性があることを学びました。

ここでも同じことを確認します。

まず、

```python
MY_DEPTH = ___
```

に自分で値を入れてください。

例えば、

- 2
- 3
- 5
- 10

などを試してください。


In [ ]:
# ↓↓↓ 自分で値を入力してください ↓↓↓
MY_DEPTH = ___

my_tree = DecisionTreeRegressor(
    max_depth=MY_DEPTH,
    random_state=42
)

my_tree.fit(
    X_train,
    y_train
)

train_pred_my_tree = my_tree.predict(
    X_train
)

test_pred_my_tree = my_tree.predict(
    X_test
)

train_r2 = r2_score(
    y_train,
    train_pred_my_tree
)

test_r2 = r2_score(
    y_test,
    test_pred_my_tree
)

print("max_depth:", MY_DEPTH)
print(f"Train R2: {train_r2:.3f}")
print(f"Test R2 : {test_r2:.3f}")


### ミニ演習3：depthを変えて記録する

| max_depth | Train R² | Test R² |
|---|---|---|
| 2 | | |
| 3 | | |
| 5 | | |
| 10 | | |

### 考えてみよう

1. max_depthを大きくすると、Train R²はどうなりましたか？
2. Test R²も同じように上がり続けましたか？
3. 2-1で学んだ過学習と同じ現象が見えましたか？

答え：

1.
2.
3.


---
## 15. Decision Treeが重要と考えた特徴量を見る

Decision Treeでは、各特徴量の重要度を確認できます。

ただし、

> Feature importanceが高い = その特徴量が疾患進行の原因である

という意味ではありません。

これは**モデルの予測にどれだけ使われたか**を表す指標です。


In [ ]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": tree_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=importance_df,
    x="importance",
    y="feature"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Decision Tree Feature Importance")
plt.show()


### ミニ演習4：重要度を解釈する

1. 最も重要度が高い特徴量は何ですか？
2. 相関係数が大きかった特徴量と一致していますか？
3. Feature importanceから因果関係を結論できますか？

答え：

1.
2.
3.


---
## 16. 最後のミニ考察

ここまでの結果を使って、短くまとめてください。

### Q1. ClassificationとRegressionの違い

答え：

### Q2. 今回のデータでは、Linear RegressionとDecision Treeのどちらが良いと考えましたか？

答え：

理由：

### Q3. Decision Treeのmax_depthを大きくすると何が起こりましたか？

答え：

### Q4. このモデルを実際の研究で使うなら、次に何を確認したいですか？

例：

- 別のデータセットでも性能を確認する
- 他のモデルと比較する
- 重要特徴量の生物学的意味を調べる
- より多くの症例で検証する

あなたの考え：


---
## 17. まとめ

2日目では、機械学習とAIの基本的な流れを体験しました。

### 2-1

**乳がんデータ × Classification**

- Logistic Regression
- Decision Tree
- Treeの可視化
- 判定経路
- max_depth
- 過学習

### 2-2

**MNIST画像 × CNN**

- 画像を数値として扱う
- CNN
- 学習曲線
- 予測確率
- Confusion Matrix
- 誤分類
- Grad-CAM

### 2-3

**糖尿病データ × Regression**

- Xとy
- Train/Test split
- fit
- predict
- MAE
- RMSE
- R²
- Linear Regression
- Decision Tree Regressor
- Feature importance
- 過学習

---

## 最後に

機械学習では、モデルを動かすことだけが目的ではありません。

重要なのは、

1. どのような問いを解きたいか考える
2. データを確認する
3. Training dataとTest dataを分ける
4. モデルを学習する
5. 適切な指標で評価する
6. 結果を解釈する
7. 未知データでも使えるか考える

という一連の流れです。

この2日間で体験した解析は、生命科学研究でも広く応用できます。
